# 05 - Cloud Artifacts and Cosmos DB

Document the low-consumption Azure Cosmos DB Serverless target and the local policy artifacts that can be promoted later.

## Review Boundaries

- This notebook does not read `.env`.
- This notebook does not print keys, tokens, or connection strings.
- Azure CLI checks are optional and use metadata-only commands.

In [ ]:
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent

policy_artifact_dir = repo_root / 'reports' / 'policy_training'
cloud_setup_doc = repo_root / 'docs' / 'cloud-setup.md'

{
    'policy_artifact_dir_exists': policy_artifact_dir.exists(),
    'cloud_setup_doc_exists': cloud_setup_doc.exists(),
}

## Cosmos DB Serverless Target

In [ ]:
cosmos_setup = {
    'resource_group': 'FIAPTechChallange5',
    'account': 'ecloe5cosmos1266cl',
    'api': 'Cosmos DB for NoSQL',
    'capacity_mode': 'Serverless',
    'region': 'Chile Central',
    'database': 'ecloe',
    'endpoint': 'https://ecloe5cosmos1266cl.documents.azure.com:443/',
    'containers': {
        'decisions': {'partition_key': '/customer_id'},
        'rewards': {'partition_key': '/customer_id'},
        'policy_versions': {'partition_key': '/policy_name'},
    },
    'required_cloud_auth': {
        'AUTH_MODE': 'entra_id',
        'AZURE_COSMOS_AUTH_MODE': 'managed_identity',
        'DECISION_REPOSITORY_MODE': 'cosmos',
    },
}
cosmos_setup


## Published Training Results

The validated training results were published to Cosmos DB with:

```bash
python scripts/publish_training_results_to_cosmos.py
```

Destination: database `ecloe`, container `policy_versions`, partition key `/policy_name`.

Confirmed counts after publication: `policy_versions: 5`, `decisions: 0`, `rewards: 0`. This is expected because training results are stored in `policy_versions`; `decisions` and `rewards` are operational containers populated only by runtime API events.


In [ ]:
published_training_results = {
    'command': 'python scripts/publish_training_results_to_cosmos.py',
    'database': 'ecloe',
    'container': 'policy_versions',
    'partition_key': '/policy_name',
    'run_id': 'train-20260727T211636Z-43b893e',
    'document_counts': {
        'policy_versions': 5,
        'decisions': 0,
        'rewards': 0,
    },
    'published_documents': {
        'policy_version': [
            'baseline',
            'epsilon_greedy',
            'ucb',
            'thompson_sampling',
        ],
        'training_run': 1,
    },
}
published_training_results


### Published Document Shape

`policy_versions` stores one document per evaluated policy and one run-level document. The partition key is always `policy_name`, allowing policy-specific lookups without mixing training metadata into runtime event containers.


In [ ]:
policy_version_document_shape = {
    'id': '<run_id>:<policy_name>:offline-v1',
    'document_type': 'policy_version',
    'policy_name': '<policy_name>',
    'policy_version': 'offline-v1',
    'run_id': '<run_id>',
    'selected': False,
    'metrics': {
        'rounds': 17232,
        'cumulative_reward': 252,
        'conversion_rate': 0.014624,
        'cumulative_regret': 0.0,
        'exploration_rate': 0.0,
    },
    'artifact_manifest': {
        'run_id': '<run_id>',
        'dataset_sha256': '<sha256>',
        'artifact_manifest_path': 'reports/policy_training/artifact_manifest.json',
    },
}

training_run_document_shape = {
    'id': '<run_id>:training_run',
    'document_type': 'training_run',
    'policy_name': '__training_run__',
    'run_id': '<run_id>',
    'selected_policy': 'baseline',
    'published_policy_count': 4,
    'artifact_manifest': {'run_id': '<run_id>'},
}

{
    'policy_version_document_shape': policy_version_document_shape,
    'training_run_document_shape': training_run_document_shape,
}


### Optional Count Verification

This optional cell uses Azure identity and the Cosmos SDK. It does not read `.env` and it does not print keys, tokens, or connection strings. Run it only after `az login` and after the signed-in identity has Cosmos DB data-plane access.


In [ ]:
# Optional verification cell. Requires azure-cosmos and azure-identity.
# It intentionally uses a literal endpoint and DefaultAzureCredential instead of reading .env.
#
# from azure.cosmos import CosmosClient
# from azure.identity import DefaultAzureCredential
#
# endpoint = 'https://ecloe5cosmos1266cl.documents.azure.com:443/'
# database_name = 'ecloe'
# containers = ['policy_versions', 'decisions', 'rewards']
#
# client = CosmosClient(endpoint, credential=DefaultAzureCredential())
# database = client.get_database_client(database_name)
# counts = {}
# for container_name in containers:
#     container = database.get_container_client(container_name)
#     rows = list(container.query_items(
#         query='SELECT VALUE COUNT(1) FROM c',
#         enable_cross_partition_query=True,
#     ))
#     counts[container_name] = rows[0]
#
# counts


## ECloe Pay Azure SQL Login Seed

ECloe Pay login personas can be prepared from a local unmasked XLSX seed file and imported into Azure SQL without storing plaintext passwords in the cloud.

Local-only generation:

```bash
python -m scripts.seed_ecloe_pay_login_xlsx --generate
```

The generated `data/demo/ecloe_pay_login_seed.local.xlsx` file contains simulated unmasked profile data and plaintext demo passwords for local preparation only. It is ignored by Git through `data/demo/*.local.xlsx`.

Azure SQL import:

```bash
python -m scripts.seed_ecloe_pay_login_xlsx --xlsx data/demo/ecloe_pay_login_seed.local.xlsx
```

The import hashes every password before writing to `ecloe_pay.demo_users`; plaintext passwords are never stored in SQL. Rich profile data is written to `ecloe_pay.demo_user_profiles`, where sensitive columns use Azure SQL Dynamic Data Masking. The routine Pay runtime identity should not receive `UNMASK`.


In [ ]:
pay_seed_xlsx = repo_root / 'data' / 'demo' / 'ecloe_pay_login_seed.local.xlsx'
pay_seed_script = repo_root / 'scripts' / 'seed_ecloe_pay_login_xlsx.py'
pay_schema = repo_root / 'src' / 'demo' / 'ecloe_pay' / 'schema.sql'

pay_login_seed_status = {
    'xlsx_path': str(pay_seed_xlsx.relative_to(repo_root)),
    'xlsx_exists_locally': pay_seed_xlsx.exists(),
    'xlsx_gitignored': 'data/demo/*.local.xlsx',
    'seed_script_exists': pay_seed_script.exists(),
    'sql_tables': ['ecloe_pay.demo_users', 'ecloe_pay.demo_user_profiles'],
    'cloud_masking': 'Azure SQL Dynamic Data Masking',
    'password_storage': 'Werkzeug password_hash only; no plaintext password in SQL',
    'runtime_unmask_permission': 'not granted',
}
pay_login_seed_status


### Optional ECloe Pay SQL Import

This optional cell requires Azure SQL dependencies, ODBC Driver 18, and an authenticated Azure identity. It does not read `.env` and the commands are commented so the notebook cannot mutate the cloud database unless explicitly edited and run.


In [ ]:
# Optional ECloe Pay Azure SQL seed commands. Uncomment only after az login and SQL access setup.
#
# import os
# os.chdir(repo_root)
# !python -m scripts.seed_ecloe_pay_login_xlsx --generate
# !python -m scripts.seed_ecloe_pay_login_xlsx --xlsx data/demo/ecloe_pay_login_seed.local.xlsx
#
# Or seed through the initializer after schema application:
# %env ECLOE_PAY_LOGIN_SEED_XLSX=data/demo/ecloe_pay_login_seed.local.xlsx
# !python -m scripts.init_ecloe_pay_sql


## Local Artifacts Ready for Promotion

In [ ]:
promotion_artifacts = [
    'selected_policy.json',
    'policy_versions.json',
    'metrics.json',
    'golden_set_recommendations.json',
    'purchase_likelihood_model.json',
    'artifact_manifest.json',
]

[
    {
        'artifact': name,
        'exists': (policy_artifact_dir / name).exists(),
        'path': str((policy_artifact_dir / name).relative_to(repo_root)),
    }
    for name in promotion_artifacts
]


## Optional Azure CLI Verification

Run these commands only in an authenticated Azure CLI session. They list account and container metadata without reading keys.

```bash
az cosmosdb show --resource-group FIAPTechChallange5 --name ecloe5cosmos1266cl --query "{name:name,location:location,kind:kind}" -o table
az cosmosdb sql container list --resource-group FIAPTechChallange5 --account-name ecloe5cosmos1266cl --database-name ecloe -o table
```

In [ ]:
# Optional metadata-only Azure CLI checks. Uncomment only after az login.
# !az cosmosdb show --resource-group FIAPTechChallange5 --name ecloe5cosmos1266cl --query "{name:name,location:location,kind:kind}" -o table
# !az cosmosdb sql container list --resource-group FIAPTechChallange5 --account-name ecloe5cosmos1266cl --database-name ecloe -o table